In [ ]:
from pathlib import Path
import requests
import subprocess

output_dir = Path("/kaggle/working/ruod_download")
output_dir.mkdir(parents=True, exist_ok=True)

api_url = (
    "https://api.github.com/repos/xiaoDetection/RUOD/"
    "releases/tags/untagged-4f7c7ab75187d68b6449"
)

response = requests.get(api_url, timeout=30)
response.raise_for_status()
release = response.json()

assets = {
    asset["name"]: asset
    for asset in release.get("assets", [])
}

for name in ["RUOD.tar.partaa", "RUOD.tar.partab"]:
    if name not in assets:
        raise RuntimeError(
            f"{name} not found. Available files: {list(assets)}"
        )

    asset = assets[name]
    destination = output_dir / name
    expected_size = asset["size"]

    if (
        destination.exists()
        and destination.stat().st_size == expected_size
    ):
        print("Already downloaded:", name)
        continue

    print("Downloading:", name, flush=True)

    subprocess.run(
        [
            "curl",
            "--fail",
            "--location",
            "--retry", "3",
            "--connect-timeout", "30",
            "--continue-at", "-",
            "--output", str(destination),
            asset["browser_download_url"]
        ],
        check=True
    )

    if destination.stat().st_size != expected_size:
        raise RuntimeError(f"Incomplete download: {name}")

    print("Complete:", name, flush=True)

print("\nBoth RUOD parts downloaded successfully.")

In [ ]:
from pathlib import Path
from collections import Counter
import shutil
import tarfile

download_dir = Path("/kaggle/working/ruod_download")
parts = [
    download_dir / "RUOD.tar.partaa",
    download_dir / "RUOD.tar.partab"
]

archive_path = download_dir / "RUOD.tar"
output_dir = Path("/kaggle/working/ruod_original")

for part in parts:
    if not part.exists():
        raise FileNotFoundError(part)

# Join the parts in the required order.
expected_size = sum(part.stat().st_size for part in parts)

if not (
    archive_path.exists()
    and archive_path.stat().st_size == expected_size
):
    print("Joining archive parts...")

    temporary = download_dir / "RUOD.tar.partial"

    with temporary.open("wb") as destination:
        for part in parts:
            with part.open("rb") as source:
                shutil.copyfileobj(
                    source, destination,
                    length=8 * 1024 * 1024
                )

    temporary.replace(archive_path)

print("Extracting...")
output_dir.mkdir(parents=True, exist_ok=True)

with tarfile.open(archive_path, mode="r:*") as archive:
    archive.extractall(path=output_dir, filter="data")

files = sorted(
    path for path in output_dir.rglob("*") if path.is_file()
)
counts = Counter(path.suffix.lower() for path in files)

print("\nEXTRACTION COMPLETE")
print("Folder:", output_dir)

print("\nFILE COUNTS")
for extension, count in counts.most_common(10):
    print(extension or "(no extension)", ":", count)

print("\nANNOTATION / CONFIG EXAMPLES")
for extension in [".xml", ".json", ".txt", ".yaml", ".yml"]:
    examples = [
        str(path.relative_to(output_dir))
        for path in files
        if path.suffix.lower() == extension
    ][:3]

    if examples:
        print(extension, ":", examples)

In [ ]:
from pathlib import Path
import re
import subprocess
import zipfile

# Clean the URL currently stored in the notebook.
raw = str(download_url).strip()
match = re.search(r"\]\((https?://[^)]+)\)", raw)
clean_url = match.group(1) if match else raw

assert clean_url.startswith("https://dataverse.no/")
assert not any(c in clean_url for c in "[]()")
print("CLEAN URL:", repr(clean_url), flush=True)

folder = Path("/kaggle/working/norfisk_download")
folder.mkdir(parents=True, exist_ok=True)
destination = folder / "NorFisk_v1.0.zip"

if destination.exists() and zipfile.is_zipfile(destination):
    print("ZIP already exists:", destination)
else:
    subprocess.run([
        "curl", "--fail", "--location",
        "--connect-timeout", "30",
        "--continue-at", "-",
        "--output", str(destination),
        clean_url,
    ], check=True)

assert zipfile.is_zipfile(destination), "Downloaded file is not a ZIP."
print("NorFisk ZIP ready:", destination)

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from collections import Counter

zip_path = Path(
    "/kaggle/working/norfisk_download/NorFisk_v1.0.zip"
)
output_dir = Path("/kaggle/working/norfisk_original")

if not zip_path.exists():
    raise FileNotFoundError(
        "এই session-এ ZIP নেই। Saved output input হিসেবে "
        "যোগ করতে হবে—আবার download এখনই কোরো না।"
    )

output_dir.mkdir(parents=True, exist_ok=True)

with ZipFile(zip_path) as archive:
    # Check paths before extraction.
    for member in archive.infolist():
        destination = (output_dir / member.filename).resolve()
        if not destination.is_relative_to(output_dir.resolve()):
            raise ValueError(f"Unsafe archive path: {member.filename}")

    print("Extracting NorFisk...")
    archive.extractall(output_dir)

files = sorted(
    path for path in output_dir.rglob("*") if path.is_file()
)
counts = Counter(path.suffix.lower() for path in files)

print("\nFILE COUNTS")
for extension, count in counts.most_common(10):
    print(extension or "(no extension)", ":", count)

print("\nEXAMPLE PATHS")
for extension in [".jpg", ".jpeg", ".png", ".csv", ".txt", ".xml", ".json"]:
    examples = [
        str(path.relative_to(output_dir))
        for path in files if path.suffix.lower() == extension
    ][:3]
    if examples:
        print(extension, ":", examples)

# Inspect a few small text/CSV annotation candidates.
candidates = [
    path for path in files
    if path.suffix.lower() in {".csv", ".txt"}
    and "annot" in str(path).lower()
]

for path in candidates[:3]:
    print("\nANNOTATION SAMPLE:", path.relative_to(output_dir))
    with path.open(errors="replace") as file:
        for _ in range(3):
            line = file.readline()
            if not line:
                break
            print(line.rstrip()[:500])

In [ ]:
from pathlib import Path
from collections import Counter
import json
import csv
import pandas as pd

report_dir = Path("/kaggle/working/three_domain_audit")
report_dir.mkdir(parents=True, exist_ok=True)

class_rows = []
file_rows = []

def inspect_coco(dataset, path):
    data = json.loads(path.read_text())

    if not (
        isinstance(data, dict)
        and all(key in data for key in [
            "images", "annotations", "categories"
        ])
    ):
        return

    categories = {
        category["id"]: category["name"]
        for category in data["categories"]
    }
    counts = Counter(
        annotation["category_id"]
        for annotation in data["annotations"]
    )

    file_rows.append({
        "dataset": dataset,
        "annotation_file": str(path),
        "image_records": len(data["images"]),
        "boxes": len(data["annotations"])
    })

    for category_id, name in categories.items():
        class_rows.append({
            "dataset": dataset,
            "annotation_file": path.name,
            "class": name,
            "boxes": counts[category_id]
        })

# Brackish: read only ground-truth JSONs.
brackish_files = [
    path for path in Path("/kaggle/input").rglob("*groundtruth.json")
    if "brackish" in str(path).lower()
]

if not brackish_files:
    raise FileNotFoundError("Brackish dataset Input-এ add করো।")

for path in sorted(brackish_files):
    inspect_coco("Brackish", path)

# RUOD: keep each JSON separate to avoid double-counting subsets.
ruod_dir = Path("/kaggle/working/ruod_original")

if not ruod_dir.exists():
    raise FileNotFoundError("Extracted RUOD folder পাওয়া যায়নি।")

for path in sorted(ruod_dir.rglob("*.json")):
    inspect_coco("RUOD", path)

# NorFisk: filename,class,xmin,ymin,xmax,ymax,width,height.
norfisk_dir = Path("/kaggle/working/norfisk_original")
norfisk_counts = Counter()
norfisk_image_names = set()
annotation_count = 0

for path in sorted(norfisk_dir.rglob("*.txt")):
    if path.parent.name.lower() != "annotations":
        continue

    annotation_count += 1

    with path.open(newline="") as file:
        for row in csv.reader(file):
            if not row or not any(value.strip() for value in row):
                continue

            if len(row) != 8:
                raise ValueError(f"Unexpected annotation: {path}: {row}")

            # Verify that coordinate fields are numeric.
            [float(value) for value in row[2:]]

            norfisk_image_names.add(row[0].strip())
            norfisk_counts[row[1].strip()] += 1

for name, count in sorted(norfisk_counts.items()):
    class_rows.append({
        "dataset": "NorFisk",
        "annotation_file": "individual TXT files",
        "class": name,
        "boxes": count
    })

file_rows.append({
    "dataset": "NorFisk",
    "annotation_file": f"{annotation_count} TXT files",
    "image_records": len(norfisk_image_names),
    "boxes": sum(norfisk_counts.values())
})

classes = pd.DataFrame(class_rows)
inventory = pd.DataFrame(file_rows)

classes.to_csv(report_dir / "class_box_counts.csv", index=False)
inventory.to_csv(report_dir / "annotation_inventory.csv", index=False)

print("ANNOTATION FILE SUMMARY")
display(inventory)

print("\nBRACKISH — totals across original splits")
display(
    classes[classes["dataset"] == "Brackish"]
    .groupby("class", as_index=False)["boxes"].sum()
)

print("\nRUOD — each annotation file shown separately")
display(classes[classes["dataset"] == "RUOD"])

print("\nNORFISK")
display(classes[classes["dataset"] == "NorFisk"])

print("\nSaved reports:", report_dir)

In [ ]:

from pathlib import Path
from PIL import Image
import csv
import math
import pandas as pd

root = Path("/kaggle/working/norfisk_original")
output = Path("/kaggle/working/three_domain_audit")
output.mkdir(parents=True, exist_ok=True)

records = []
issues = []

for annotation in sorted(root.rglob("*.txt")):
    if annotation.parent.name.lower() != "annotations":
        continue

    with annotation.open(newline="") as file:
        for row in csv.reader(file):
            if not row or not any(value.strip() for value in row):
                continue

            if len(row) != 8:
                issues.append({
                    "file": str(annotation),
                    "issue": "unexpected_row_format"
                })
                continue

            name, label = row[0].strip(), row[1].strip()
            image_path = annotation.parent.parent / "img" / name

            if not image_path.exists():
                issues.append({
                    "file": str(annotation),
                    "issue": "missing_image"
                })
                continue

            try:
                x1, y1, x2, y2, aw, ah = map(float, row[2:])

                with Image.open(image_path) as image:
                    width, height = image.size

                problems = []

                if not all(math.isfinite(v) for v in [
                    x1, y1, x2, y2, aw, ah
                ]):
                    problems.append("non_finite")
                else:
                    if (width, height) != (aw, ah):
                        problems.append("dimension_mismatch")

                    if x2 <= x1 or y2 <= y1:
                        problems.append("non_positive_box")

                    if (
                        x1 < 0 or y1 < 0
                        or x2 > width or y2 > height
                    ):
                        problems.append("out_of_bounds")

                for problem in problems:
                    issues.append({
                        "file": str(annotation),
                        "issue": problem
                    })

                records.append({
                    "image_path": str(image_path),
                    "annotation_path": str(annotation),
                    "original_class": label,
                    "width": width,
                    "height": height,
                    "xmin": x1, "ymin": y1,
                    "xmax": x2, "ymax": y2,
                    "status": "ok" if not problems else "|".join(problems)
                })

            except Exception as error:
                issues.append({
                    "file": str(annotation),
                    "issue": f"read_error: {error}"
                })

manifest = pd.DataFrame(records)
issue_table = pd.DataFrame(issues, columns=["file", "issue"])

manifest.to_csv(output / "norfisk_manifest.csv", index=False)
issue_table.to_csv(output / "norfisk_issues.csv", index=False)

print("Annotation rows matched to readable images:", len(manifest))
print("Unique matched images:", manifest["image_path"].nunique())
print("Issue entries:", len(issue_table))

if len(issue_table):
    display(issue_table["issue"].value_counts())
else:
    print("Image matching, dimensions and box boundaries: PASSED")

display(
    manifest.groupby("original_class").agg(
        images=("image_path", "nunique"),
        min_width=("width", "min"),
        max_width=("width", "max"),
        min_height=("height", "min"),
        max_height=("height", "max")
    )
)

print("Reports saved:", output)

In [ ]:
from pathlib import Path
from collections import defaultdict
from PIL import Image
import json
import pandas as pd

source = Path("/kaggle/working/ruod_original")
audit = Path("/kaggle/working/three_domain_audit")
audit.mkdir(parents=True, exist_ok=True)

lookup = defaultdict(list)

for path in source.rglob("*"):
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        lookup[path.name].append(path)

issues = []
checked = 0

for split in ["train", "test"]:
    matches = [
        path for path in source.rglob(f"instances_{split}.json")
        if path.parent.name == "RUOD_ANN"
    ]

    if len(matches) != 1:
        raise RuntimeError(f"Cannot locate {split} annotation uniquely")

    data = json.loads(matches[0].read_text())

    for record in data["images"]:
        name = Path(record["file_name"]).name
        candidates = lookup.get(name, [])

        if len(candidates) != 1:
            issues.append({
                "split": split,
                "filename": name,
                "issue": "missing_or_ambiguous_image"
            })
            continue

        with Image.open(candidates[0]) as image:
            actual_width, actual_height = image.size
            orientation = image.getexif().get(274, 1)

        checked += 1
        expected_width = record["width"]
        expected_height = record["height"]

        if (
            actual_width != expected_width
            or actual_height != expected_height
        ):
            issues.append({
                "split": split,
                "filename": name,
                "issue": "dimension_mismatch",
                "image_path": str(candidates[0]),
                "json_width": expected_width,
                "json_height": expected_height,
                "actual_width": actual_width,
                "actual_height": actual_height,
                "exif_orientation": orientation,
                "width_height_swapped": (
                    actual_width == expected_height
                    and actual_height == expected_width
                )
            })

report = pd.DataFrame(issues)
report.to_csv(audit / "ruod_dimension_issues.csv", index=False)

print("Images checked:", checked)
print("Issue entries:", len(report))

if len(report):
    display(report.head(20))
else:
    print("No dimension mismatches found.")

print("Saved:", audit / "ruod_dimension_issues.csv")

In [ ]:
from pathlib import Path
from collections import defaultdict
from PIL import Image, ImageOps
import json
import pandas as pd

source = Path("/kaggle/working/ruod_original")
audit = Path("/kaggle/working/three_domain_audit")

lookup = defaultdict(list)

for path in source.rglob("*"):
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        lookup[path.name].append(path)

resolved_rows = []
unresolved = []
duplicate_examples = []

for split in ["train", "test"]:
    json_files = [
        path for path in source.rglob(f"instances_{split}.json")
        if path.parent.name == "RUOD_ANN"
    ]
    if len(json_files) != 1:
        raise RuntimeError(f"Cannot locate {split} JSON uniquely")

    data = json.loads(json_files[0].read_text())

    for record in data["images"]:
        name = Path(record["file_name"]).name
        candidates = lookup.get(name, [])

        if len(candidates) > 1 and len(duplicate_examples) < 3:
            duplicate_examples.append({
                "filename": name,
                "paths": [str(path) for path in candidates]
            })

        # Prefer the original split inside the main RUOD_pic folder.
        preferred = [
            path for path in candidates
            if "RUOD_pic" in path.parts
            and split in path.relative_to(source).parts[:-1]
        ]

        if len(preferred) == 1:
            chosen = preferred[0]
        elif len(candidates) == 1:
            chosen = candidates[0]
        else:
            unresolved.append({
                "split": split,
                "filename": name,
                "issue": "path_not_resolved",
                "candidate_paths": [str(path) for path in candidates]
            })
            continue

        with Image.open(chosen) as image:
            raw_size = image.size
            orientation = image.getexif().get(274, 1)
            oriented_size = ImageOps.exif_transpose(image).size

        expected = (record["width"], record["height"])

        if oriented_size != expected:
            unresolved.append({
                "split": split,
                "filename": name,
                "issue": "dimensions_still_mismatch",
                "raw_size": raw_size,
                "oriented_size": oriented_size,
                "expected_size": expected
            })
            continue

        resolved_rows.append({
            "original_split": split,
            "image_id": record["id"],
            "filename": name,
            "original_path": str(chosen),
            "width": expected[0],
            "height": expected[1],
            "exif_orientation": orientation,
            "needs_exif_normalization": orientation != 1
        })

resolved = pd.DataFrame(resolved_rows)
resolved.to_csv(audit / "ruod_resolved_images.csv", index=False)

pd.DataFrame(unresolved).to_csv(
    audit / "ruod_unresolved_images.csv", index=False
)

print("Resolved images:", len(resolved))
print("Unresolved images:", len(unresolved))
print(
    "Images needing EXIF normalization:",
    int(resolved["needs_exif_normalization"].sum())
)

print("\nDuplicate filename examples:")
for example in duplicate_examples:
    print(example["filename"])
    for path in example["paths"]:
        print(" ", path)

if unresolved:
    print("\nUnresolved examples:")
    display(pd.DataFrame(unresolved).head(10))

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import json
import math

BASE = Path("/kaggle/working/lodo_ready")
AUDIT = BASE / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

all_records = []
corrections = []
exclusions = []

def export_image(domain, key, source_path, width, height,
                 boxes, group, original_split="unknown"):
    """Export one image and its fish-only labels."""
    source_path = Path(source_path)

    if not source_path.exists():
        raise FileNotFoundError(source_path)

    folder = BASE / domain
    images = folder / "images"
    labels = folder / "labels"
    images.mkdir(parents=True, exist_ok=True)
    labels.mkdir(parents=True, exist_ok=True)

    destination = images / f"{key}{source_path.suffix.lower()}"
    label_path = labels / f"{key}.txt"

    lines = []

    for box in boxes:
        x, y, w, h = map(float, box)

        if (
            not all(math.isfinite(v) for v in [x, y, w, h])
            or w <= 0 or h <= 0
        ):
            raise ValueError(f"Invalid box: {domain}/{key}: {box}")

        x1, y1 = max(0.0, x), max(0.0, y)
        x2 = min(float(width), x + w)
        y2 = min(float(height), y + h)

        if x2 <= x1 or y2 <= y1:
            raise ValueError(f"Outside box: {domain}/{key}: {box}")

        clipped = [x1, y1, x2-x1, y2-y1]

        if not np.allclose(box, clipped, atol=1e-6, rtol=0):
            corrections.append({
                "domain": domain, "image": key,
                "original": list(box), "clipped": clipped
            })

        lines.append(
            f"0 {(x1+x2)/(2*width):.8f} "
            f"{(y1+y2)/(2*height):.8f} "
            f"{(x2-x1)/width:.8f} {(y2-y1)/height:.8f}"
        )

    # Links avoid duplicating several GB in this session.
    if destination.is_symlink():
        if destination.resolve() != source_path.resolve():
            raise RuntimeError(f"Wrong existing link: {destination}")
    elif destination.exists():
        raise RuntimeError(f"Unexpected existing file: {destination}")
    else:
        destination.symlink_to(source_path.resolve())

    label_path.write_text("\n".join(lines) + ("\n" if lines else ""))

    all_records.append({
        "domain": domain,
        "key": key,
        "image_path": str(destination),
        "label_path": str(label_path),
        "source_path": str(source_path),
        "width": width,
        "height": height,
        "fish_boxes": len(lines),
        "group": str(group),
        "original_split": original_split
    })

# Use the NorFisk manifest already verified in Cell 9.
nf = pd.read_csv(
    "/kaggle/working/three_domain_audit/norfisk_manifest.csv"
)

if not nf["status"].eq("ok").all():
    raise RuntimeError("NorFisk manifest contains unresolved issues.")

for row in nf.itertuples(index=False):
    path = Path(row.image_path)
    export_image(
        domain="norfisk",
        key=path.stem,
        source_path=path,
        width=row.width,
        height=row.height,
        boxes=[[
            row.xmin, row.ymin,
            row.xmax-row.xmin, row.ymax-row.ymin
        ]],
        # Individual image identity; not a recording ID.
        group=path.stem
    )

print("NorFisk exported:", len(nf), "images")

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

usage = shutil.disk_usage("/kaggle/working")
print(f"Total storage: {usage.total / 2**30:.2f} GB")
print(f"Used storage:  {usage.used / 2**30:.2f} GB")
print(f"Free storage:  {usage.free / 2**30:.2f} GB")

archive_dir = Path("/kaggle/working/ruod_download")
rows = []

if archive_dir.exists():
    for p in sorted(archive_dir.iterdir()):
        if p.is_file():
            rows.append({
                "file": str(p),
                "size_GB": round(p.stat().st_size / 2**30, 3)
            })

display(pd.DataFrame(rows))

frame_dir = Path("/kaggle/working/lodo_ready/brackish_frames")
print(
    "Brackish PNG files currently present:",
    sum(1 for _ in frame_dir.glob("*.png"))
)

In [ ]:
from pathlib import Path
import shutil

# Confirm that extracted RUOD data is present.
ruod_root = Path("/kaggle/working/ruod_original/RUOD")

jpg_count = sum(1 for _ in ruod_root.rglob("*.jpg"))
main_annotations = list(ruod_root.rglob("instances_train.json"))
test_annotations = list(ruod_root.rglob("instances_test.json"))

assert jpg_count >= 14000, f"Unexpected RUOD image count: {jpg_count}"
assert main_annotations and test_annotations, "RUOD annotations missing."

# Remove only redundant downloaded archive files.
archive_dir = Path("/kaggle/working/ruod_download")

for name in ["RUOD.tar", "RUOD.tar.partaa", "RUOD.tar.partab"]:
    p = archive_dir / name
    if p.is_file():
        size = p.stat().st_size / 2**30
        p.unlink()
        print(f"Removed: {name} ({size:.2f} GB)")

# Remove the PNG whose write failed, so extraction recreates it.
failed_frame = Path(
    "/kaggle/working/lodo_ready/brackish_frames/"
    "2019-03-21_03-21-54to2019-03-21_03-22-09_1-0120.png"
)
failed_frame.unlink(missing_ok=True)

print(
    f"\nFree storage: "
    f"{shutil.disk_usage('/kaggle/working').free / 2**30:.2f} GB"
)

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path("/kaggle/working/lodo_ready")
AUDIT = BASE / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

manifest_path = AUDIT / "all_images.csv"

print("Manifest exists:", manifest_path.is_file())
print("Export function available:", callable(globals().get("export_image")))
print("In-memory records:", len(globals().get("all_records", [])))

for domain in ["brackish", "ruod", "norfisk"]:
    image_dir = BASE / domain / "images"
    label_dir = BASE / domain / "labels"

    print(
        domain,
        "| readable images:", sum(p.is_file() for p in image_dir.glob("*")),
        "| labels:", sum(p.is_file() for p in label_dir.glob("*.txt")),
    )

if manifest_path.is_file():
    manifest = pd.read_csv(manifest_path)
    print("\nManifest columns:", manifest.columns.tolist())
    print("Duplicate keys:", manifest.duplicated(["domain", "key"]).sum())
    display(manifest.groupby("domain").size().rename("images"))

In [ ]:
from pathlib import Path
from collections import defaultdict
from PIL import Image
import cv2
import json
import pandas as pd
import shutil

BASE = Path("/kaggle/working/lodo_ready")
AUDIT = BASE / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

# Keep the already exported NorFisk records.
# This also makes rerunning this cell safe for the in-memory manifest.
all_records = [
    r for r in all_records if r["domain"] == "norfisk"
]
assert len(all_records) == 12514, "Run the NorFisk export cell first."

# ---------- RUOD ----------
ruod_source = Path("/kaggle/working/ruod_original")
resolved_path = Path(
    "/kaggle/working/three_domain_audit/ruod_resolved_images.csv"
)
assert resolved_path.is_file(), "Run the RUOD path-resolution cell first."

resolved = pd.read_csv(resolved_path)
lookup = {
    (r.original_split, r.filename): r
    for r in resolved.itertuples(index=False)
}

for split in ["train", "test"]:
    matches = [
        p for p in ruod_source.rglob(f"instances_{split}.json")
        if p.parent.name == "RUOD_ANN"
    ]
    assert len(matches) == 1, f"Cannot locate RUOD {split} annotations."

    data = json.loads(matches[0].read_text())
    fish_ids = {
        c["id"] for c in data["categories"]
        if c["name"].strip().lower() == "fish"
    }

    boxes = defaultdict(list)
    for ann in data["annotations"]:
        if ann["category_id"] in fish_ids:
            boxes[ann["image_id"]].append(ann["bbox"])

    for record in data["images"]:
        filename = Path(record["file_name"]).name

        # Previously identified orientation/box-alignment issue.
        if filename == "008431.jpg":
            exclusions.append({
                "domain": "ruod",
                "image": filename,
                "reason": "Unresolved orientation/annotation alignment",
            })
            continue

        row = lookup[(split, filename)]
        orientation = int(row.exif_orientation)
        assert orientation in [0, 1], (
            f"Review orientation: {filename}, EXIF={orientation}"
        )

        source = Path(row.original_path)
        with Image.open(source) as image:
            actual_size = image.size

        assert actual_size == (record["width"], record["height"]), (
            f"Dimension mismatch: {filename}"
        )

        key = f"{split}_{Path(filename).stem}"
        export_image(
            domain="ruod",
            key=key,
            source_path=source,
            width=record["width"],
            height=record["height"],
            boxes=boxes[record["id"]],
            group=key,
            original_split=split,
        )

    print("RUOD exported:", split, flush=True)

# ---------- BRACKISH ----------
matches = [
    p for p in Path("/kaggle/input").rglob("train_groundtruth.json")
    if "brackish" in str(p).lower()
]
assert len(matches) == 1, "Add the Brackish dataset to notebook inputs."

annotation_dir = matches[0].parent
brackish_root = annotation_dir.parent.parent

records_by_name = {}
fish_boxes = defaultdict(list)

for split in ["train", "valid", "test"]:
    data = json.loads(
        (annotation_dir / f"{split}_groundtruth.json").read_text()
    )
    fish_ids = {
        c["id"] for c in data["categories"]
        if c["name"].strip().lower() in {"fish", "small_fish"}
    }
    id_to_name = {}

    for record in data["images"]:
        filename = Path(record["file_name"]).name
        assert filename not in records_by_name, (
            f"Repeated Brackish image record: {filename}"
        )
        records_by_name[filename] = (record, split)
        id_to_name[record["id"]] = filename

    for ann in data["annotations"]:
        if ann["category_id"] in fish_ids:
            fish_boxes[id_to_name[ann["image_id"]]].append(ann["bbox"])

videos = defaultdict(list)
for p in brackish_root.rglob("*.avi"):
    videos[p.stem].append(p)

frames_by_video = defaultdict(dict)
for filename in records_by_name:
    video_id, number = Path(filename).stem.rsplit("-", 1)
    frames_by_video[video_id][int(number)] = filename

frame_dir = BASE / "brackish_frames"
frame_dir.mkdir(parents=True, exist_ok=True)

for index, (video_id, wanted) in enumerate(
    sorted(frames_by_video.items()), start=1
):
    assert len(videos[video_id]) == 1, f"Video missing/ambiguous: {video_id}"

    pending = {}
    for number, filename in wanted.items():
        destination = frame_dir / filename

        if destination.exists():
            existing = cv2.imread(str(destination))
            if existing is not None and existing.shape[:2] == (540, 960):
                continue
            destination.unlink()

        pending[number] = filename

    if pending:
        capture = cv2.VideoCapture(str(videos[video_id][0]))
        assert capture.isOpened(), f"Cannot open video: {video_id}"
        number = 0

        try:
            while pending:
                ok, frame = capture.read()
                if not ok:
                    break
                number += 1

                if number not in pending:
                    continue

                resized = cv2.resize(
                    frame, (960, 540), interpolation=cv2.INTER_CUBIC
                )
                destination = frame_dir / pending[number]

                if not cv2.imwrite(str(destination), resized):
                    destination.unlink(missing_ok=True)
                    raise RuntimeError(
                        "Frame write failed. Free storage: "
                        f"{shutil.disk_usage(BASE).free / 2**30:.2f} GB"
                    )

                del pending[number]
        finally:
            capture.release()

        assert not pending, f"Missing frames in {video_id}: {len(pending)}"

    for filename in wanted.values():
        record, split = records_by_name[filename]
        assert (record["width"], record["height"]) == (960, 540)

        export_image(
            domain="brackish",
            key=Path(filename).stem,
            source_path=frame_dir / filename,
            width=960,
            height=540,
            boxes=fish_boxes[filename],
            group=video_id[:10],  # Recording date
            original_split=split,
        )

    if index % 10 == 0 or index == len(frames_by_video):
        print(
            f"Brackish videos: {index}/{len(frames_by_video)}",
            flush=True,
        )

# ---------- SAVE MANIFEST AND LOGS ----------
manifest = pd.DataFrame(all_records)
assert not manifest.duplicated(["domain", "key"]).any()

assert manifest.groupby("domain").size().to_dict() == {
    "brackish": 12444,
    "norfisk": 12514,
    "ruod": 13999,
}

manifest.to_csv(AUDIT / "all_images.csv", index=False)

def save_log(records, filename):
    unique = {}
    for record in records:
        signature = json.dumps(record, sort_keys=True, default=str)
        unique[signature] = record
    pd.DataFrame(unique.values()).to_csv(AUDIT / filename, index=False)

save_log(corrections, "box_corrections.csv")
save_log(exclusions, "excluded_images.csv")

display(
    manifest.groupby("domain").agg(
        images=("key", "size"),
        fish_boxes=("fish_boxes", "sum"),
    )
)
print("EXPORT COMPLETE — manifest saved.")
print(
    "Free storage:",
    round(shutil.disk_usage(BASE).free / 2**30, 2),
    "GB",
)

In [ ]:
import hashlib
from sklearn.model_selection import GroupShuffleSplit
import yaml

manifest = pd.read_csv(AUDIT / "all_images.csv")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

hashes = []
for index, path in enumerate(manifest["source_path"], start=1):
    hashes.append(sha256_file(path))
    if index % 5000 == 0:
        print("Files hashed:", index)

manifest["sha256"] = hashes

cross_domain = (
    manifest.groupby("sha256")["domain"].nunique()
)
overlap_hashes = cross_domain[cross_domain > 1].index

manifest.to_csv(AUDIT / "all_images_hashed.csv", index=False)

if len(overlap_hashes):
    manifest[manifest["sha256"].isin(overlap_hashes)].to_csv(
        AUDIT / "cross_domain_duplicates.csv", index=False
    )
    raise RuntimeError(
        "Cross-domain duplicates found. Review before training."
    )

manifest["source_partition"] = ""

for domain in ["brackish", "ruod", "norfisk"]:
    domain_rows = manifest[manifest["domain"] == domain]

    if domain == "brackish":
        # Union date groups connected by any exact duplicate.
        parent = {group: group for group in domain_rows["group"].unique()}

        def find(group):
            while parent[group] != group:
                parent[group] = parent[parent[group]]
                group = parent[group]
            return group

        for _, duplicates in domain_rows.groupby("sha256"):
            groups = duplicates["group"].unique()
            for group in groups[1:]:
                parent[find(group)] = find(groups[0])

        split_groups = domain_rows["group"].map(find).to_numpy()
    else:
        split_groups = domain_rows["sha256"].to_numpy()

    splitter = GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=42
    )
    train_index, val_index = next(
        splitter.split(domain_rows, groups=split_groups)
    )

    manifest.loc[
        domain_rows.index[train_index], "source_partition"
    ] = "train"
    manifest.loc[
        domain_rows.index[val_index], "source_partition"
    ] = "val"

    train_hashes = set(domain_rows.iloc[train_index]["sha256"])
    val_hashes = set(domain_rows.iloc[val_index]["sha256"])
    assert train_hashes.isdisjoint(val_hashes)

manifest.to_csv(AUDIT / "fixed_partitions.csv", index=False)

config_dir = BASE / "configs"
config_dir.mkdir(parents=True, exist_ok=True)

runs = [
    ("brackish_ruod_to_norfisk", ["brackish", "ruod"], "norfisk"),
    ("ruod_norfisk_to_brackish", ["ruod", "norfisk"], "brackish"),
    ("norfisk_brackish_to_ruod", ["norfisk", "brackish"], "ruod")
]

for name, sources, target in runs:
    train = manifest[
        manifest["domain"].isin(sources)
        & manifest["source_partition"].eq("train")
    ]
    val = manifest[
        manifest["domain"].isin(sources)
        & manifest["source_partition"].eq("val")
    ]
    test = manifest[manifest["domain"].eq(target)]

    lists = {}
    for split, table in [("train", train), ("val", val), ("test", test)]:
        path = config_dir / f"{name}_{split}.txt"
        path.write_text("\n".join(table["image_path"]) + "\n")
        lists[split] = str(path)

    config = {
        "path": str(BASE),
        **lists,
        "names": {0: "fish"}
    }

    (config_dir / f"{name}.yaml").write_text(
        yaml.safe_dump(config, sort_keys=False)
    )

    print(
        f"{name}: train={len(train)}, "
        f"source_val={len(val)}, unseen_test={len(test)}"
    )

display(
    manifest.groupby(["domain", "source_partition"]).agg(
        images=("key", "size"),
        fish_boxes=("fish_boxes", "sum")
    )
)

print("\nConfigs saved:", config_dir)
print("PREPARATION COMPLETE — no model trained yet.")

## Training 

In [ ]:
%pip install -q ultralytics==8.4.207

In [ ]:
from pathlib import Path
import json
import hashlib
import shutil
import yaml
import torch
from ultralytics import YOLO
from IPython.display import display, FileLink

BASE = Path("/kaggle/working/lodo_ready")
RUN_NAME = "ruod_norfisk_to_brackish"
CONFIG = BASE / "configs" / f"{RUN_NAME}.yaml"
RUN_DIR = Path("/kaggle/working/lodo_runs") / RUN_NAME

assert torch.cuda.is_available(), "GPU enable করো।"
assert CONFIG.is_file(), f"Config missing: {CONFIG}"
assert not RUN_DIR.exists(), (
    f"Run folder already exists: {RUN_DIR}\n"
    "আগের run overwrite না করে আগে সেটির status দেখো।"
)

cfg = yaml.safe_load(CONFIG.read_text())
data_root = Path(cfg.get("path") or CONFIG.parent)
if not data_root.is_absolute():
    data_root = CONFIG.parent / data_root

def read_image_list(field):
    value = cfg.get(field)
    assert isinstance(value, str), f"{field}: expected one TXT list"

    list_path = Path(value)
    if not list_path.is_absolute():
        list_path = data_root / list_path

    assert list_path.is_file(), f"Missing list: {list_path}"
    paths = []

    for line in list_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        p = Path(line)
        if not p.is_absolute():
            p = list_path.parent / p
        paths.append(p)

    assert paths, f"Empty {field} list"
    missing = [str(p) for p in paths if not p.is_file()]
    assert not missing, f"{field}: missing images, examples: {missing[:3]}"
    return paths

train_images = read_image_list("train")
val_images = read_image_list("val")
test_images = read_image_list("test")

def domain_of(p):
    matches = {
        d for d in ("brackish", "ruod", "norfisk")
        if d in p.parts
    }
    assert len(matches) == 1, f"Cannot identify domain: {p}"
    return matches.pop()

assert {domain_of(p) for p in train_images} == {"ruod", "norfisk"}
assert {domain_of(p) for p in val_images} <= {"ruod", "norfisk"}
assert {domain_of(p) for p in test_images} == {"brackish"}

assert not (
    {str(p.resolve()) for p in train_images}
    & {str(p.resolve()) for p in val_images}
), "Train/validation path overlap"

print("GPU:", torch.cuda.get_device_name(0))
print("Training images:", len(train_images))
print("Source validation images:", len(val_images))
print("Reserved Brackish test images:", len(test_images))
print("Brackish is excluded from training and validation.", flush=True)

# Fresh initialization, NOT the first experiment's best.pt.
model = YOLO("yolo11n.pt")

model.train(
    data=str(CONFIG),
    project="/kaggle/working/lodo_runs",
    name=RUN_NAME,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    seed=42,
    deterministic=True,
    patience=30,
    save=True,
    save_period=5,
    plots=True,
    exist_ok=False,
)

# Small backup: trained model, training reports and exact config/lists.
REPORT = Path("/kaggle/working/run2_training_report")
shutil.copytree(RUN_DIR, REPORT, dirs_exist_ok=True)
shutil.copy2(CONFIG, REPORT / CONFIG.name)

for field in ("train", "val", "test"):
    (REPORT / f"{field}_images.txt").write_text(
        "\n".join(str(p) for p in {
            "train": train_images,
            "val": val_images,
            "test": test_images,
        }[field]) + "\n"
    )

best = RUN_DIR / "weights/best.pt"
assert best.is_file(), "Training finished but best.pt not found."

(REPORT / "checkpoint_identity.json").write_text(json.dumps({
    "run": RUN_NAME,
    "best_sha256": hashlib.sha256(best.read_bytes()).hexdigest(),
    "target_test_performed": False,
}, indent=2))

archive = shutil.make_archive(
    "/kaggle/working/run2_training_report",
    "zip",
    root_dir=REPORT,
)

print("\nTRAINING COMPLETE")
print("Best model:", best)
print("Backup:", archive)
display(FileLink(archive))

## backup

In [ ]:
from pathlib import Path
from IPython.display import display, FileLink
import shutil

RUN = Path("/kaggle/working/lodo_runs/ruod_norfisk_to_brackish")
BACKUP = Path("/kaggle/working/run2_training_report.zip")

assert (RUN / "weights/best.pt").is_file(), "best.pt missing"

if not BACKUP.is_file():
    shutil.make_archive(
        "/kaggle/working/run2_training_report",
        "zip",
        root_dir=RUN,
    )

print("Training backup:", round(BACKUP.stat().st_size / 1024**2, 2), "MB")
display(FileLink(str(BACKUP)))

# Separate model download as an additional backup.
shutil.copy2(RUN / "weights/best.pt", "/kaggle/working/run2_best.pt")
display(FileLink("/kaggle/working/run2_best.pt"))

In [ ]:
from pathlib import Path
import shutil
from IPython.display import display, FileLink

RUN = Path("/kaggle/working/lodo_runs/ruod_norfisk_to_brackish")
TEST = Path("/kaggle/working/run2_brackish_test_report")
OUT = Path("/kaggle/working")

files = {
    RUN / "results.csv": "run2_training_results.csv",
    RUN / "args.yaml": "run2_training_settings.yaml",
    RUN / "results.png": "run2_training_curves.png",
    TEST / "unseen_test_metrics.csv": "run2_test_metrics.csv",
    TEST / "evaluation_details.json": "run2_test_details.json",
}

for source, name in files.items():
    if source.is_file():
        destination = OUT / name
        shutil.copy2(source, destination)
        print(name, f"— {destination.stat().st_size / 1024:.1f} KB")
        display(FileLink(str(destination)))
    else:
        print("Not available yet:", name)

## Brackish unseen test

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display, FileLink
import torch
import pandas as pd
import json
import hashlib
import shutil
import yaml
from zipfile import ZipFile, ZIP_DEFLATED

RUN = Path("/kaggle/working/lodo_runs/ruod_norfisk_to_brackish")
CONFIG = Path(
    "/kaggle/working/lodo_ready/configs/ruod_norfisk_to_brackish.yaml"
)
WEIGHTS = RUN / "weights/best.pt"
REPORT = Path("/kaggle/working/run2_brackish_test_report")

assert WEIGHTS.is_file(), "best.pt missing"
assert CONFIG.is_file(), "Dataset config missing"
assert torch.cuda.is_available(), "GPU is required"

# Confirm the test list contains only Brackish images.
cfg = yaml.safe_load(CONFIG.read_text())
data_root = Path(cfg.get("path") or CONFIG.parent)
if not data_root.is_absolute():
    data_root = CONFIG.parent / data_root

test_list = Path(cfg["test"])
if not test_list.is_absolute():
    test_list = data_root / test_list

assert test_list.is_file(), f"Missing test list: {test_list}"

images = []
for line in test_list.read_text().splitlines():
    if line.strip():
        p = Path(line.strip())
        if not p.is_absolute():
            p = test_list.parent / p
        images.append(p)

assert images and all("brackish" in p.parts for p in images)
assert len(set(map(str, images))) == len(images), "Duplicate test paths"

records = []
for p in images:
    assert p.is_file(), f"Missing test image: {p}"
    assert p.parent.name == "images", f"Unexpected layout: {p}"
    label = p.parent.parent / "labels" / f"{p.stem}.txt"
    assert label.is_file(), f"Missing label: {label}"
    records.append({
        "image_id": p.stem,
        "image_path": str(p),
        "label_path": str(label),
        "fish_boxes": sum(
            bool(row.strip()) for row in label.read_text().splitlines()
        ),
    })

print("Unseen Brackish images:", len(records))
print("Fish boxes:", sum(r["fish_boxes"] for r in records), flush=True)

model = YOLO(str(WEIGHTS))
metrics = model.val(
    data=str(CONFIG),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    conf=0.001,
    iou=0.7,
    max_det=300,
    augment=False,
    plots=True,
    save_json=True,
    save_txt=True,
    save_conf=True,
    project="/kaggle/working/lodo_tests",
    name="ruod_norfisk_to_brackish",
    exist_ok=False,
)

REPORT.mkdir(parents=True, exist_ok=True)

table = pd.DataFrame([{
    "Training domains": "RUOD + NorFisk",
    "Unseen test domain": "Brackish",
    "Test images": len(records),
    "Fish boxes": sum(r["fish_boxes"] for r in records),
    "Precision (%)": float(metrics.box.mp) * 100,
    "Recall (%)": float(metrics.box.mr) * 100,
    "mAP50 (%)": float(metrics.box.map50) * 100,
    "mAP50-95 (%)": float(metrics.box.map) * 100,
}])

# Save metrics immediately, before packaging.
table.to_csv(REPORT / "unseen_test_metrics.csv", index=False)
pd.DataFrame(records).to_csv(REPORT / "test_image_mapping.csv", index=False)

(REPORT / "evaluation_details.json").write_text(json.dumps({
    "model_sha256": hashlib.sha256(WEIGHTS.read_bytes()).hexdigest(),
    "metrics_results_dict": {
        k: float(v) for k, v in metrics.results_dict.items()
    },
    "speed_ms_per_image": metrics.speed,
    "settings": {
        "split": "test", "imgsz": 640, "batch": 16,
        "conf": 0.001, "iou": 0.7, "max_det": 300,
        "augment": False,
    },
}, indent=2, default=str))

shutil.copytree(metrics.save_dir, REPORT / "evaluation", dirs_exist_ok=True)
shutil.copy2(WEIGHTS, REPORT / "best.pt")
shutil.copy2(CONFIG, REPORT / "dataset_config.yaml")

for name in ["args.yaml", "results.csv"]:
    if (RUN / name).is_file():
        shutil.copy2(RUN / name, REPORT / f"training_{name}")

# Ground-truth labels support later error analysis without retraining.
with ZipFile(REPORT / "ground_truth_labels.zip", "w", ZIP_DEFLATED) as z:
    for r in records:
        z.write(r["label_path"], arcname=f'{r["image_id"]}.txt')

archive = shutil.make_archive(str(REPORT), "zip", root_dir=REPORT)

display(table.round(3))
print("\nDownload and keep:", archive)
display(FileLink(archive))

In [ ]:
from pathlib import Path
import yaml
import torch
import shutil
from ultralytics import YOLO
from IPython.display import display, FileLink

BASE = Path("/kaggle/working/lodo_ready")
RUN_NAME = "norfisk_brackish_to_ruod"
CONFIG = BASE / "configs" / f"{RUN_NAME}.yaml"
RUN_DIR = Path("/kaggle/working/lodo_runs") / RUN_NAME

assert torch.cuda.is_available(), "GPU required"
assert CONFIG.is_file(), f"Missing config: {CONFIG}"
assert not RUN_DIR.exists(), f"Run already exists: {RUN_DIR}"

cfg = yaml.safe_load(CONFIG.read_text())
data_root = Path(cfg.get("path") or CONFIG.parent)

def load_paths(split):
    listing = Path(cfg[split])
    if not listing.is_absolute():
        listing = data_root / listing

    assert listing.is_file(), f"Missing list: {listing}"
    paths = []
    for line in listing.read_text().splitlines():
        if line.strip():
            p = Path(line.strip())
            if not p.is_absolute():
                p = listing.parent / p
            paths.append(p)

    assert paths, f"Empty {split}"
    assert all(p.is_file() for p in paths), f"Missing {split} images"
    return paths

train = load_paths("train")
val = load_paths("val")
test = load_paths("test")

def domain(p):
    matches = set(p.parts) & {"norfisk", "brackish", "ruod"}
    assert len(matches) == 1, f"Ambiguous domain: {p}"
    return matches.pop()

assert {domain(p) for p in train} == {"norfisk", "brackish"}
assert {domain(p) for p in val} <= {"norfisk", "brackish"}
assert {domain(p) for p in test} == {"ruod"}
assert not (
    {str(p.resolve()) for p in train}
    & {str(p.resolve()) for p in val}
), "Train/validation path overlap"

print("Training images:", len(train))
print("Source validation:", len(val))
print("Reserved unseen RUOD:", len(test), flush=True)

# Independent run with fresh pretrained initialization.
model = YOLO("yolo11n.pt")

model.train(
    data=str(CONFIG),
    project="/kaggle/working/lodo_runs",
    name=RUN_NAME,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    seed=42,
    deterministic=True,
    patience=30,
    save=True,
    save_period=5,
    plots=True,
    exist_ok=False,
)

# Backup training artifacts, without dataset images.
BACKUP = Path("/kaggle/working/run3_training_report")
shutil.copytree(RUN_DIR, BACKUP, dirs_exist_ok=True)
shutil.copy2(CONFIG, BACKUP / CONFIG.name)

for split in ["train", "val", "test"]:
    listing = Path(cfg[split])
    if not listing.is_absolute():
        listing = data_root / listing
    shutil.copy2(listing, BACKUP / f"{split}_images.txt")

archive = shutil.make_archive(str(BACKUP), "zip", root_dir=BACKUP)

shutil.copy2(
    RUN_DIR / "weights/best.pt",
    "/kaggle/working/run3_best.pt",
)

print("\nRUN 3 TRAINING COMPLETE")
display(FileLink(archive))
display(FileLink("/kaggle/working/run3_best.pt"))

## backup

In [ ]:
from pathlib import Path
import shutil
from IPython.display import display, FileLink

RUN = Path("/kaggle/working/lodo_runs/norfisk_brackish_to_ruod")
assert (RUN / "weights/best.pt").is_file(), "best.pt missing"

# Individual files, useful if ZIP download fails.
for relative, output_name in {
    "weights/best.pt": "run3_best.pt",
    "weights/last.pt": "run3_last.pt",
    "results.csv": "run3_training_results.csv",
    "args.yaml": "run3_training_settings.yaml",
    "results.png": "run3_training_curves.png",
}.items():
    source = RUN / relative
    if source.is_file():
        destination = Path("/kaggle/working") / output_name
        shutil.copy2(source, destination)
        display(FileLink(str(destination)))

archive = Path("/kaggle/working/run3_training_report.zip")
if not archive.is_file():
    shutil.make_archive(
        "/kaggle/working/run3_training_report",
        "zip",
        root_dir=RUN,
    )

display(FileLink(str(archive)))

## unseen test

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from ultralytics import YOLO
from IPython.display import display, FileLink
import torch
import yaml
import pandas as pd
import json
import hashlib
import shutil

RUN = Path("/kaggle/working/lodo_runs/norfisk_brackish_to_ruod")
CONFIG = Path(
    "/kaggle/working/lodo_ready/configs/norfisk_brackish_to_ruod.yaml"
)
WEIGHTS = RUN / "weights/best.pt"
REPORT = Path("/kaggle/working/run3_ruod_test_report")

assert WEIGHTS.is_file(), "best.pt missing"
assert CONFIG.is_file(), "Config missing"
assert torch.cuda.is_available(), "GPU required"

cfg = yaml.safe_load(CONFIG.read_text())
data_root = Path(cfg.get("path") or CONFIG.parent)
if not data_root.is_absolute():
    data_root = CONFIG.parent / data_root

listing = Path(cfg["test"])
if not listing.is_absolute():
    listing = data_root / listing
assert listing.is_file(), f"Missing test list: {listing}"

records = []
for line in listing.read_text().splitlines():
    if not line.strip():
        continue

    image = Path(line.strip())
    if not image.is_absolute():
        image = listing.parent / image

    assert "ruod" in image.parts, f"Unexpected test domain: {image}"
    assert image.is_file(), f"Missing image: {image}"
    assert image.parent.name == "images", f"Unexpected layout: {image}"

    label = image.parent.parent / "labels" / f"{image.stem}.txt"
    assert label.is_file(), f"Missing label: {label}"

    records.append({
        "image_id": image.stem,
        "image_path": str(image),
        "label_path": str(label),
        "fish_boxes": sum(
            bool(row.strip()) for row in label.read_text().splitlines()
        ),
    })

assert len(records) == 13999, f"Unexpected image count: {len(records)}"
assert len({r["image_id"] for r in records}) == len(records)
box_count = sum(r["fish_boxes"] for r in records)
assert box_count == 13091, f"Unexpected box count: {box_count}"

print("Unseen RUOD images:", len(records))
print("Fish boxes:", box_count, flush=True)

model = YOLO(str(WEIGHTS))
metrics = model.val(
    data=str(CONFIG),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    conf=0.001,
    iou=0.7,
    max_det=300,
    augment=False,
    plots=True,
    save_json=True,
    save_txt=True,
    save_conf=True,
    project="/kaggle/working/lodo_tests",
    name="norfisk_brackish_to_ruod",
    exist_ok=False,
)

REPORT.mkdir(parents=True, exist_ok=True)

table = pd.DataFrame([{
    "Training domains": "NorFisk + Brackish",
    "Unseen test domain": "RUOD",
    "Test images": len(records),
    "Fish boxes": box_count,
    "Precision (%)": float(metrics.box.mp) * 100,
    "Recall (%)": float(metrics.box.mr) * 100,
    "mAP50 (%)": float(metrics.box.map50) * 100,
    "mAP50-95 (%)": float(metrics.box.map) * 100,
}])

table.to_csv(REPORT / "unseen_test_metrics.csv", index=False)
pd.DataFrame(records).to_csv(REPORT / "test_image_mapping.csv", index=False)

(REPORT / "evaluation_details.json").write_text(json.dumps({
    "model_sha256": hashlib.sha256(WEIGHTS.read_bytes()).hexdigest(),
    "metrics_results_dict": {
        k: float(v) for k, v in metrics.results_dict.items()
    },
    "speed_ms_per_image": metrics.speed,
    "settings": {
        "split": "test", "imgsz": 640, "batch": 16,
        "conf": 0.001, "iou": 0.7, "max_det": 300,
        "augment": False,
    },
}, indent=2, default=str))

shutil.copytree(metrics.save_dir, REPORT / "evaluation", dirs_exist_ok=True)
shutil.copy2(WEIGHTS, REPORT / "best.pt")
shutil.copy2(CONFIG, REPORT / "dataset_config.yaml")

for name in ["args.yaml", "results.csv"]:
    if (RUN / name).is_file():
        shutil.copy2(RUN / name, REPORT / f"training_{name}")

with ZipFile(REPORT / "ground_truth_labels.zip", "w", ZIP_DEFLATED) as z:
    for row in records:
        z.write(row["label_path"], arcname=f'{row["image_id"]}.txt')

# Copy small metrics files to working root for separate downloads.
for name in ["unseen_test_metrics.csv", "evaluation_details.json"]:
    destination = Path("/kaggle/working") / f"run3_{name}"
    shutil.copy2(REPORT / name, destination)

archive = shutil.make_archive(str(REPORT), "zip", root_dir=REPORT)

display(table.round(3))
display(FileLink(archive))
display(FileLink("/kaggle/working/run3_unseen_test_metrics.csv"))
display(FileLink("/kaggle/working/run3_evaluation_details.json"))

In [ ]:
from pathlib import Path
import shutil
from IPython.display import display, FileLink

REPORT = Path("/kaggle/working/run3_ruod_test_report")
OUT = Path("/kaggle/working")

files = {
    REPORT / "evaluation/predictions.json":
        "run3_predictions.json",
    REPORT / "test_image_mapping.csv":
        "run3_test_image_mapping.csv",
    REPORT / "ground_truth_labels.zip":
        "run3_ground_truth_labels.zip",
}

for source, name in files.items():
    if source.is_file():
        destination = OUT / name
        shutil.copy2(source, destination)
        print(name, f"| {destination.stat().st_size / 1024**2:.2f} MB")
        display(FileLink(str(destination)))
    else:
        print("MISSING:", source)

# Export existing label/prediction preview images separately.
evaluation = REPORT / "evaluation"
for source in sorted(evaluation.glob("val_batch*.jpg"))[:6]:
    destination = OUT / f"run3_{source.name}"
    shutil.copy2(source, destination)
    display(FileLink(str(destination)))

## original annotations আর exported fish labels

In [32]:
from pathlib import Path
from collections import defaultdict
from PIL import Image
from IPython.display import display, FileLink
import pandas as pd
import numpy as np
import json

BASE = Path("/kaggle/working/lodo_ready")
SOURCE = Path("/kaggle/working/ruod_original")
MANIFEST = BASE / "audit/all_images.csv"

assert MANIFEST.is_file(), f"Missing: {MANIFEST}"
manifest = pd.read_csv(MANIFEST)
ruod = manifest[manifest["domain"] == "ruod"].copy()

assert len(ruod) == 13999, f"Unexpected RUOD count: {len(ruod)}"
assert not ruod["key"].duplicated().any()

expected = {}
excluded = []
clipped_boxes = 0

for split in ["train", "test"]:
    matches = [
        p for p in SOURCE.rglob(f"instances_{split}.json")
        if p.parent.name == "RUOD_ANN"
    ]
    assert len(matches) == 1, f"{split} annotations: {matches}"

    data = json.loads(matches[0].read_text())
    fish_ids = {
        c["id"] for c in data["categories"]
        if c["name"].strip().lower() == "fish"
    }
    assert fish_ids, "Fish category missing"

    annotations = defaultdict(list)
    for annotation in data["annotations"]:
        if annotation["category_id"] in fish_ids:
            annotations[annotation["image_id"]].append(annotation)

    for image in data["images"]:
        filename = Path(image["file_name"]).name
        key = f"{split}_{Path(filename).stem}"

        # Previously documented orientation/box exclusion.
        if filename == "008431.jpg":
            excluded.append(key)
            continue

        width, height = image["width"], image["height"]
        boxes = []

        for annotation in annotations[image["id"]]:
            x, y, w, h = map(float, annotation["bbox"])
            assert np.isfinite([x, y, w, h]).all()
            assert w > 0 and h > 0

            x1 = np.clip(x, 0, width)
            y1 = np.clip(y, 0, height)
            x2 = np.clip(x + w, 0, width)
            y2 = np.clip(y + h, 0, height)

            if not np.allclose(
                [x1, y1, x2, y2], [x, y, x + w, y + h]
            ):
                clipped_boxes += 1

            if x2 > x1 and y2 > y1:
                boxes.append([
                    0,
                    (x1 + x2) / (2 * width),
                    (y1 + y2) / (2 * height),
                    (x2 - x1) / width,
                    (y2 - y1) / height,
                ])

        expected[key] = {
            "width": width,
            "height": height,
            "boxes": boxes,
        }

issues = []
checked_boxes = 0

def canonical(rows):
    return np.asarray(
        sorted(tuple(round(float(v), 7) for v in row) for row in rows),
        dtype=float,
    ).reshape(-1, 5)

assert set(ruod["key"]) == set(expected), \
    "Original/exported image keys differ"

for number, row in enumerate(ruod.itertuples(index=False), 1):
    reference = expected[row.key]
    image_path = Path(row.image_path)
    label_path = Path(row.label_path)

    if not image_path.is_file() or not label_path.is_file():
        issues.append({"key": row.key, "issue": "missing_image_or_label"})
        continue

    try:
        with Image.open(image_path) as image:
            size = image.size
        if size != (reference["width"], reference["height"]):
            issues.append({
                "key": row.key,
                "issue": "image_dimensions_differ_from_original_JSON",
                "actual": str(size),
                "expected": str(
                    (reference["width"], reference["height"])
                ),
            })
    except Exception as error:
        issues.append({
            "key": row.key,
            "issue": "image_read_error",
            "detail": str(error),
        })
        continue

    try:
        actual = [
            list(map(float, line.split()))
            for line in label_path.read_text().splitlines()
            if line.strip()
        ]
        wanted = canonical(reference["boxes"])
        actual = canonical(actual)
        checked_boxes += len(actual)

        if actual.shape != wanted.shape:
            issues.append({
                "key": row.key,
                "issue": "fish_box_count_mismatch",
                "actual": len(actual),
                "expected": len(wanted),
            })
        elif not np.allclose(actual, wanted, atol=2e-6, rtol=0):
            issues.append({
                "key": row.key,
                "issue": "fish_box_coordinates_or_class_mismatch",
            })
    except Exception as error:
        issues.append({
            "key": row.key,
            "issue": "label_parse_error",
            "detail": str(error),
        })

    if number % 3000 == 0:
        print(f"Checked {number}/{len(ruod)}", flush=True)

summary = {
    "images_checked": len(ruod),
    "exported_fish_boxes_read": checked_boxes,
    "expected_fish_boxes": sum(len(v["boxes"]) for v in expected.values()),
    "issue_entries": len(issues),
    "original_boxes_requiring_clipping": clipped_boxes,
    "documented_exclusions": excluded,
    "scope": "Conversion consistency; not annotation completeness or leakage",
}

summary_path = Path("/kaggle/working/run3_ruod_conversion_audit.json")
issues_path = Path("/kaggle/working/run3_ruod_conversion_issues.csv")

summary_path.write_text(json.dumps(summary, indent=2))
pd.DataFrame(
    issues if issues else [],
    columns=["key", "issue", "actual", "expected", "detail"],
).to_csv(issues_path, index=False)

print("\nAUDIT SUMMARY")
print(json.dumps(summary, indent=2))

if issues:
    display(pd.DataFrame(issues).head(20))
else:
    print("Original JSON → exported labels: PASSED")

display(FileLink(str(summary_path)))
display(FileLink(str(issues_path)))

Checked 3000/13999
Checked 6000/13999
Checked 9000/13999
Checked 12000/13999

AUDIT SUMMARY
{
  "images_checked": 13999,
  "exported_fish_boxes_read": 13091,
  "expected_fish_boxes": 13091,
  "issue_entries": 0,
  "original_boxes_requiring_clipping": 0,
  "documented_exclusions": [
    "train_008431"
  ],
  "scope": "Conversion consistency; not annotation completeness or leakage"
}
Original JSON → exported labels: PASSED


/kaggle/working/run3_ruod_conversion_audit.json

/kaggle/working/run3_ruod_conversion_issues.csv

## false positives ও missed fish গণনা করব

In [33]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, FileLink

GT = Path("/kaggle/working/lodo_ready/ruod/labels")
EVAL = Path("/kaggle/working/lodo_tests/norfisk_brackish_to_ruod")
PRED = EVAL / "labels"

CONF = 0.25
MATCH_IOU = 0.50

assert GT.is_dir(), "Ground-truth labels missing"
assert PRED.is_dir(), "Saved prediction labels missing"
assert (EVAL / "predictions.json").is_file(), "Prediction JSON missing"

gt_files = sorted(GT.glob("*.txt"))
assert len(gt_files) == 13999, f"Unexpected labels: {len(gt_files)}"

def read_rows(path, columns):
    if not path.is_file() or not path.read_text().strip():
        return np.empty((0, columns))
    rows = np.array([
        list(map(float, line.split()))
        for line in path.read_text().splitlines()
        if line.strip()
    ])
    assert rows.ndim == 2 and rows.shape[1] == columns, path
    assert np.isfinite(rows).all(), path
    assert (rows[:, 0] == 0).all(), f"Unexpected class: {path}"
    return rows

def corners(boxes):
    result = boxes.copy()
    result[:, :2] = boxes[:, :2] - boxes[:, 2:] / 2
    result[:, 2:] = boxes[:, :2] + boxes[:, 2:] / 2
    return result

def iou_matrix(pred, truth):
    if not len(pred) or not len(truth):
        return np.zeros((len(pred), len(truth)))
    p, g = corners(pred), corners(truth)
    lower = np.maximum(p[:, None, :2], g[None, :, :2])
    upper = np.minimum(p[:, None, 2:], g[None, :, 2:])
    intersection = np.maximum(upper - lower, 0).prod(axis=2)
    pa = np.maximum(p[:, 2:] - p[:, :2], 0).prod(axis=1)
    ga = np.maximum(g[:, 2:] - g[:, :2], 0).prod(axis=1)
    return intersection / np.maximum(
        pa[:, None] + ga[None, :] - intersection, 1e-12
    )

records = []

for i, label in enumerate(gt_files, 1):
    gt = read_rows(label, 5)
    # A missing prediction TXT normally means no saved detections.
    pred = read_rows(PRED / label.name, 6)
    pred = pred[pred[:, 5] >= CONF]
    pred = pred[np.argsort(-pred[:, 5])]

    overlaps = iou_matrix(pred[:, 1:5], gt[:, 1:5])
    used = set()
    matched_ious = []

    for row in overlaps:
        candidates = [
            j for j in range(len(gt))
            if j not in used and row[j] >= MATCH_IOU
        ]
        if candidates:
            j = max(candidates, key=lambda j: row[j])
            used.add(j)
            matched_ious.append(float(row[j]))

    tp = len(used)
    records.append({
        "image_id": label.stem,
        "group": "fish_present" if len(gt) else "no_annotated_fish",
        "gt_fish": len(gt),
        "predictions_at_conf_025": len(pred),
        "tp": tp,
        "fp": len(pred) - tp,
        "fn": len(gt) - tp,
        "mean_matched_iou": (
            np.mean(matched_ious) if matched_ious else np.nan
        ),
    })

    if i % 3000 == 0:
        print(f"Analyzed {i}/{len(gt_files)}", flush=True)

per_image = pd.DataFrame(records)
assert per_image["gt_fish"].sum() == 13091

summary = []
for name, subset in [
    ("all", per_image),
    *list(per_image.groupby("group")),
]:
    tp, fp, fn = [
        int(subset[c].sum()) for c in ["tp", "fp", "fn"]
    ]
    summary.append({
        "group": name,
        "images": len(subset),
        "TP": tp, "FP": fp, "FN": fn,
        "precision_percent": (
            100 * tp / (tp + fp) if tp + fp else np.nan
        ),
        "recall_percent": (
            100 * tp / (tp + fn) if tp + fn else np.nan
        ),
        "FP_per_image": fp / len(subset),
        "images_with_FP": int((subset["fp"] > 0).sum()),
    })

table = pd.DataFrame(summary)
per_image.to_csv("/kaggle/working/run3_per_image_errors.csv", index=False)
table.to_csv("/kaggle/working/run3_error_summary.csv", index=False)

print(f"\nConfidence={CONF}; matching IoU={MATCH_IOU}")
display(table.round(3))
display(FileLink("/kaggle/working/run3_error_summary.csv"))
display(FileLink("/kaggle/working/run3_per_image_errors.csv"))

Analyzed 3000/13999
Analyzed 6000/13999
Analyzed 9000/13999
Analyzed 12000/13999

Confidence=0.25; matching IoU=0.5


,group,images,TP,FP,FN,precision_percent,recall_percent,FP_per_image,images_with_FP
0,all,13999,594,9217,12497,6.054,4.537,0.658,4597
1,fish_present,3165,594,2956,12497,16.732,4.537,0.934,1111
2,no_annotated_fish,10834,0,6261,0,0.000,NaN,0.578,3486


/kaggle/working/run3_error_summary.csv

/kaggle/working/run3_per_image_errors.csv

## Run 2-এর model দিয়ে একই RUOD images evaluate করবে

In [34]:
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display, FileLink
import pandas as pd
import torch
import json
import hashlib

WEIGHTS = Path(
    "/kaggle/working/lodo_runs/"
    "ruod_norfisk_to_brackish/weights/best.pt"
)
CONFIG = Path(
    "/kaggle/working/lodo_ready/configs/"
    "norfisk_brackish_to_ruod.yaml"
)
REPORT = Path("/kaggle/working/ruod_seen_model_diagnostic")
REPORT.mkdir(parents=True, exist_ok=True)

assert WEIGHTS.is_file(), (
    "Run 2 best.pt missing. Download করা Run 2 model input হিসেবে "
    "যোগ করে WEIGHTS path বদলাও। Run 3 best.pt ব্যবহার করবে না।"
)
assert CONFIG.is_file(), f"Missing config: {CONFIG}"
assert torch.cuda.is_available(), "GPU required"

model = YOLO(str(WEIGHTS))

metrics = model.val(
    data=str(CONFIG),       # Same RUOD target list used for Run 3.
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    conf=0.001,
    iou=0.7,
    max_det=300,
    augment=False,
    plots=True,
    save_json=True,
    save_txt=True,
    save_conf=True,
    project=str(REPORT),
    name="evaluation",
    exist_ok=False,
)

table = pd.DataFrame([
    {
        "Model": "Run 3: NorFisk + Brackish",
        "RUOD exposure": "Unseen during training/selection",
        "mAP50 (%)": 0.874,
        "mAP50-95 (%)": 0.348,
    },
    {
        "Model": "Run 2: RUOD + NorFisk",
        "RUOD exposure": "Seen; diagnostic only",
        "mAP50 (%)": float(metrics.box.map50) * 100,
        "mAP50-95 (%)": float(metrics.box.map) * 100,
    },
])

csv_path = REPORT / "diagnostic_comparison.csv"
table.to_csv(csv_path, index=False)

details = {
    "purpose": "Pipeline diagnostic, NOT independent test performance",
    "warning": "Evaluation includes RUOD training/validation images.",
    "checkpoint": str(WEIGHTS),
    "checkpoint_sha256": hashlib.sha256(
        WEIGHTS.read_bytes()
    ).hexdigest(),
    "dataset_config": str(CONFIG),
    "evaluation_directory": str(metrics.save_dir),
    "metrics": {
        k: float(v) for k, v in metrics.results_dict.items()
    },
    "settings": {
        "split": "test",
        "imgsz": 640,
        "batch": 16,
        "conf": 0.001,
        "iou": 0.7,
        "max_det": 300,
        "augment": False,
    },
}
json_path = REPORT / "diagnostic_details.json"
json_path.write_text(json.dumps(details, indent=2))

# Small individual files at working root for easier download.
import shutil

for source in [csv_path, json_path]:
    destination = Path("/kaggle/working") / source.name
    shutil.copy2(source, destination)
    display(FileLink(str(destination)))

display(table.round(3))

Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2366.9±972.6 MB/s, size: 115.3 KB)
val: Scanning /kaggle/working/lodo_ready/ruod/labels.cache... 13999 images, 10834 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13999/13999 3.9Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 875/875 6.7it/s 2:110.1ssss
                   all      13999      13091      0.775      0.686      0.775      0.532
Speed: 0.6ms preprocess, 2.7ms inference, 0.0ms loss, 0.8ms postprocess per image
Saving /kaggle/working/ruod_seen_model_diagnostic/evaluation/predictions.json...

Evaluating faster-coco-eval mAP...
Evaluate annotation type *bbox*
COCOeval_opt.evaluate() finished...
DONE (t=1.77s).
Accumulating evaluation results...
COCOeval_opt.accumulate() finished...
DONE (t=0.00s).
 

/kaggle/working/diagnostic_comparison.csv

/kaggle/working/diagnostic_details.json

,Model,RUOD exposure,mAP50 (%),mAP50-95 (%)
0,Run 3: NorFisk + Brackish,Unseen during training/selection,0.874,0.348
1,Run 2: RUOD + NorFisk,Seen; diagnostic only,77.485,53.214


## শুধু RUOD source-validation images evaluate

In [35]:
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display, FileLink
import pandas as pd
import yaml
import json
import hashlib

BASE = Path("/kaggle/working/lodo_ready")
RUN2 = Path("/kaggle/working/lodo_runs/ruod_norfisk_to_brackish")
OUT = Path("/kaggle/working/run2_ruod_source_validation")
OUT.mkdir(parents=True, exist_ok=True)

weights = RUN2 / "weights/best.pt"
original_config = BASE / "configs/ruod_norfisk_to_brackish.yaml"
assert weights.is_file() and original_config.is_file()

cfg = yaml.safe_load(original_config.read_text())
root = Path(cfg.get("path") or original_config.parent)
if not root.is_absolute():
    root = original_config.parent / root

listing = Path(cfg["val"])
if not listing.is_absolute():
    listing = root / listing

images = []
for line in listing.read_text().splitlines():
    if line.strip():
        p = Path(line.strip())
        if not p.is_absolute():
            p = listing.parent / p
        if "ruod" in p.parts:
            assert p.is_file(), f"Missing: {p}"
            images.append(p)

assert images, "No RUOD validation images found."
assert len(set(map(str, images))) == len(images)

val_list = OUT / "ruod_source_val_images.txt"
val_list.write_text("\n".join(map(str, images)) + "\n")

config = OUT / "ruod_source_val.yaml"
config.write_text(yaml.safe_dump({
    "path": str(OUT),
    "train": str(val_list),
    "val": str(val_list),
    "names": {0: "fish"},
}))

print("RUOD source-validation images:", len(images), flush=True)

metrics = YOLO(str(weights)).val(
    data=str(config),
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    conf=0.001,
    iou=0.7,
    max_det=300,
    augment=False,
    plots=True,
    save_json=True,
    project=str(OUT),
    name="evaluation",
    exist_ok=False,
)

table = pd.DataFrame([{
    "Model": "Run 2: RUOD + NorFisk",
    "Evaluation": "RUOD source validation; used for model selection",
    "Images": len(images),
    "Precision (%)": float(metrics.box.mp) * 100,
    "Recall (%)": float(metrics.box.mr) * 100,
    "mAP50 (%)": float(metrics.box.map50) * 100,
    "mAP50-95 (%)": float(metrics.box.map) * 100,
}])

csv = Path("/kaggle/working/run2_ruod_source_validation.csv")
details = Path("/kaggle/working/run2_ruod_source_validation_details.json")
table.to_csv(csv, index=False)
details.write_text(json.dumps({
    "checkpoint_sha256": hashlib.sha256(weights.read_bytes()).hexdigest(),
    "image_list": str(val_list),
    "evaluation_directory": str(metrics.save_dir),
    "warning": "Source validation, not an independent test.",
    "settings": {
        "imgsz": 640, "conf": 0.001, "iou": 0.7,
        "max_det": 300, "augment": False,
    },
}, indent=2))

display(table.round(3))
display(FileLink(str(csv)))
display(FileLink(str(details)))

RUOD source-validation images: 2780
Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2469.1±1414.3 MB/s, size: 180.2 KB)
val: Scanning /kaggle/working/lodo_ready/ruod/labels... 2780 images, 2176 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2780/2780 1.5Kit/s 1.8s0.1s
val: New cache created: /kaggle/working/lodo_ready/ruod/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 174/174 7.2it/s 24.3s0.1ss
                   all       2780       2436       0.73      0.642      0.723      0.484
Speed: 0.6ms preprocess, 2.7ms inference, 0.0ms loss, 0.8ms postprocess per image
Saving /kaggle/working/run2_ruod_source_validation/evaluation/predictions.json...

Evaluating faster-coco-eval mAP...
Evaluate annotation type *bbox*
COCOeval_opt.evaluate() finished...
DONE (

,Model,Evaluation,Images,Precision (%),Recall (%),mAP50 (%),mAP50-95 (%)
0,Run 2: RUOD + NorFisk,RUOD source validation; used for model selection,2780,72.971,64.167,72.294,48.432


/kaggle/working/run2_ruod_source_validation.csv

/kaggle/working/run2_ruod_source_validation_details.json

## BACKUP

In [36]:
from pathlib import Path
import json
from IPython.display import display, FileLink

EVAL = Path("/kaggle/working/lodo_tests/norfisk_brackish_to_ruod")
GT = Path("/kaggle/working/lodo_ready/ruod/labels")
OUT = Path("/kaggle/working/run3_analysis_backup.json")

assert (EVAL / "predictions.json").is_file()
assert GT.is_dir()

backup = {
    "predictions": json.loads(
        (EVAL / "predictions.json").read_text()
    ),
    "ground_truth_yolo": {
        p.stem: p.read_text()
        for p in sorted(GT.glob("*.txt"))
    },
    "saved_analysis_csv": {},
}

for name in [
    "run3_error_summary.csv",
    "run3_per_image_errors.csv",
    "run3_size_error_summary.csv",
    "run3_object_size_errors.csv",
]:
    p = Path("/kaggle/working") / name
    if p.is_file():
        backup["saved_analysis_csv"][name] = p.read_text()

OUT.write_text(json.dumps(backup))
print("Backup size:", round(OUT.stat().st_size / 1024**2, 2), "MB")
display(FileLink(str(OUT)))

Backup size: 58.42 MB


/kaggle/working/run3_analysis_backup.json

In [37]:
from pathlib import Path

names = [
    "run3_error_summary.csv",
    "run3_per_image_errors.csv",
    "run3_size_error_summary.csv",
    "run3_object_size_errors.csv",
    "run2_ruod_source_validation.csv",
]

for name in names:
    p = Path("/kaggle/working") / name
    if p.is_file():
        print(f"\n===== {name} =====")
        if p.stat().st_size < 20000:
            print(p.read_text())
        else:
            print("Large file — included in run3_analysis_backup.json "
                  "if that backup cell was run after this file was created.")


===== run3_error_summary.csv =====
group,images,TP,FP,FN,precision_percent,recall_percent,FP_per_image,images_with_FP
all,13999,594,9217,12497,6.0544287024768115,4.537468489802154,0.6584041717265519,4597
fish_present,3165,594,2956,12497,16.732394366197184,4.537468489802154,0.9339652448657187,1111
no_annotated_fish,10834,0,6261,0,0.0,,0.5779028982831825,3486


===== run3_per_image_errors.csv =====
Large file — included in run3_analysis_backup.json if that backup cell was run after this file was created.

===== run2_ruod_source_validation.csv =====
Model,Evaluation,Images,Precision (%),Recall (%),mAP50 (%),mAP50-95 (%)
Run 2: RUOD + NorFisk,RUOD source validation; used for model selection,2780,72.97061798213592,64.16721652692799,72.2938208442687,48.43152601390412

